<a href="https://colab.research.google.com/github/Sakinat-Folorunso/OOU_CSC309_Artificial_Intelligence/blob/main/notebooks/CSC309_Week12_Integration_Mock_Practical_Student_Centred.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CSC 309 – Artificial Intelligence
**Week 12 Practical (PH):** Integration — campus assistant, dialect-bias audit, responsible-AI checklist and mock practical

**Instructor:** Dr Sakinat Folorunso · Associate Professor of AI Systems · Department of Computer Sciences, Olabisi Onabanjo University (OOU), Ago-Iwoye, Ogun State, Nigeria  
**Course:** CSC 309 (2 units; LH 15, PH 45) · **Mode:** student-centred, hands-on, Google Colab  
**CCMAS learning outcomes covered:** LO1–LO8

> 🎓 **Instructor note** – this notebook is the *STUDENT (TODO cells are stubs)* version.


## How to use this notebook
1. Fill in the **Group Log**, then do the **Do Now** with your partner.
2. Run the **Setup** cell once (*Runtime → Run all* is safe: nothing here crashes if a TODO is still empty).
3. Work top to bottom. Only edit cells marked **`# TODO(Student)`**.
4. Each task ends with an **auto-check** (✅/❌). Keep going until it is ✅.
5. Finish with the **Reflection**. Fast finishers: try the **Extensions**.

6. **Read the comments.** Every line of code has a plain-English comment above it. Read the comment first, predict what the line does, then run the cell and check your prediction.
7. Hand in the **Assignment** (marks and deadline are in the notebook) after the lab.

*Every random draw uses `SEED = 42`, so your numbers should match your classmates' numbers.*


## Lab tasks at a glance

| Part | Lab task(s) you must complete |
|---|---|
| Part A — A campus assistant: spelling → retrieval → abstention | *Guided lab: run every cell, read the comments, explain the output in your own words* |
| Part B — Dialect-bias audit: English vs Nigerian Pidgin | *Guided lab: run every cell, read the comments, explain the output in your own words* |
| Part C — Responsible-AI checklist for a proposed system | *Guided lab: run every cell, read the comments, explain the output in your own words* |
| Part D — Mock practical (60 minutes, closed-book except course notebooks) | *Guided lab: run every cell, read the comments, explain the output in your own words* |
| **Assignment** | see **📝 Assignment / deliverables** near the end |
| **Reflection** | short written reflection at the end (5–7 sentences) |
| *Extensions* | optional, for fast finishers |

In [ ]:
#@title 🧑🏽‍🤝‍🧑🏾 Group Log (fill before you start)
# Form field: type your group members' names here
group_members = "Type names here"   #@param {type:"string"}
# Form field: assign roles (driver, navigator, recorder, presenter)
roles_notes   = "Driver / Navigator / Recorder / Presenter"  #@param {type:"string"}
# Print them so your notebook records who did what
print("👥 Group:", group_members); print("📝 Roles:", roles_notes)


In [ ]:
#@title 🔧 Setup (run once) — seed, logging, helpers
# Standard-library modules used by every lab (os, re, math, random, logging, time ...)
import sys, os, re, math, random, logging, subprocess, warnings, time, itertools
# NumPy: fast arrays and random numbers
import numpy as np
# One seed for the whole course so every run gives the same numbers
SEED = 42                                    # one seed for the whole course → reproducible results
# Seed both Python's and NumPy's random generators
random.seed(SEED); np.random.seed(SEED)
# Configure logging so messages print as "LEVEL | message"
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s", force=True)
# A named logger for this course
log = logging.getLogger("csc309")
# Hide noisy library warnings so output stays readable
warnings.filterwarnings("ignore")

def ensure(pip_name, import_name=None):
    """Import a package; pip-install it first if it is missing (Colab-friendly)."""
    # Try to import the library
    try:
        # Return the imported module
        return __import__(import_name or pip_name)
    # If it is not installed ...
    except ImportError:
        # ... tell the student we are installing it
        log.info("installing %s ...", pip_name)
        # ... and install it with pip
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])
        # Return the imported module
        return __import__(import_name or pip_name)

# List that stores every check result
_CHECKS = []
def check(name, condition, hint=""):
    """Auto-check. Never raises, so 'Run all' always finishes. Pass a lambda to catch your own bugs."""
    # Try to import the library
    try:
        # Run the check: call it if it is a function, otherwise use the value
        ok = bool(condition() if callable(condition) else condition)
        # Default: no error message
        err = ""
    # If the student's code crashes, we report instead of stopping the notebook
    except Exception as e:                    # student code may crash; report instead of stopping
        # Mark as failed and keep the error message
        ok, err = False, f"{type(e).__name__}: {e}"
    # Store the result
    _CHECKS.append((name, ok))
    # Print a ✅ or ❌ line with a hint on failure
    print(("✅ PASS       " if ok else "❌ TRY AGAIN  ") + name + ("" if ok else f"   → {err or hint}"))
    # Return whether the check passed
    return ok

# Print how many checks passed
def checks_summary():
    # Count the passed checks
    passed = sum(ok for _, ok in _CHECKS)
    # Print the total
    print(f"\nChecks passed: {passed}/{len(_CHECKS)}")

def mean_ci(x, conf=0.95):
    """Mean, sample std and a two-sided t-based confidence interval."""
    # SciPy for the t-distribution
    from scipy import stats
    # Convert to a float array and count the observations
    x = np.asarray(x, dtype=float); n = len(x)
    # Mean and sample standard deviation (n-1 divisor)
    m = x.mean(); s = x.std(ddof=1) if n > 1 else 0.0
    # Half-width of the 95% confidence interval from the t-distribution
    h = stats.t.ppf((1 + conf) / 2, n - 1) * s / math.sqrt(n) if n > 1 else 0.0
    # Return mean, SD, lower and upper confidence limits
    return m, s, m - h, m + h

# Print a one-line statistical summary
def report(label, x):
    # Compute the statistics
    m, s, lo, hi = mean_ci(x)
    # Print mean, SD, CI and n
    print(f"{label:<28} mean={m:8.2f}  sd={s:7.2f}  95% CI=[{lo:.2f}, {hi:.2f}]  (n={len(x)})")

def paired_wilcoxon(a, b):
    """Two-sided Wilcoxon signed-rank test for paired samples (same seeds/instances)."""
    # SciPy for the t-distribution
    from scipy import stats
    # Convert both samples to arrays
    a, b = np.asarray(a, float), np.asarray(b, float)
    # If the two samples are identical the test is undefined
    if np.allclose(a, b):
        # Report p = 1 (no difference)
        return 1.0
    # Paired Wilcoxon signed-rank test p-value
    return float(stats.wilcoxon(a, b).pvalue)

# Setup finished
print("✅ Setup complete — seed =", SEED)


### Learning objectives (with Bloom level)
| # | By the end of this lab you can… | Bloom | CCMAS LO |
|---|---|---|---|
| 1 | connect several course ideas (NLP, retrieval, thresholds, evaluation) in **one pipeline** | Apply / Create | LO7, LO8 |
| 2 | choose a decision threshold on a **dev set** and report test accuracy with a **CI** | Evaluate | LO4, LO7 |
| 3 | **audit** a model for performance gaps between English and Nigerian Pidgin | Evaluate | LO7 |
| 4 | complete a **responsible-AI checklist** for a proposed system | Evaluate / Create | LO3 |
| 5 | rehearse the practical exam under timed conditions | Apply | LO1–LO8 |

**Course map (what you can now do).**
| Week | Topic | CCMAS LO |
|---|---|---|
| 1 | AI definitions, Turing test, Chinese Room, human-like vs rational | 1, 3 |
| 2 | Intelligent agents, PEAS, environment properties | 2 |
| 3 | Uninformed search, combinatorial explosion | 5 |
| 4 | Heuristics, A\*, admissibility, completeness–optimality–time–space trade-offs | 4 |
| 5 | Predicate logic, clause form, unification, resolution | 7 |
| 6 | Semantic networks and frames | 6 |
| 7 | NLP: TF-IDF, retrieval QA, NL → logic, spam | 7 |
| 8 | Sentiment, spelling, MT, speech recognition | 7 |
| 9 | Expert systems and chatbots | 8 |
| 10 | AI languages, vision, face verification, robots | 8 |
| 11 | Agent simulations: boids, ants, message passing | 2, 5, 8 |
| 12 | Integration and responsible AI | all |

**Do Now (3 min).** Pick one item from the table that you could still not explain to a classmate in 60 seconds. That is your revision target.

In [ ]:
#@title 📦 Imports
# Plotting library
import matplotlib.pyplot as plt
# SciPy statistics
from scipy import stats
# TF-IDF vectoriser for the FAQ search
from sklearn.feature_extraction.text import TfidfVectorizer

## Part A — A campus assistant: spelling → retrieval → abstention
**Pipeline.** (1) lower-case and tokenise; (2) **correct spelling** against the FAQ vocabulary (edit distance ≤ 1 — Week 8); (3) **retrieve** the best FAQ answer by TF-IDF cosine similarity (Week 7); (4) if the best cosine score is below a threshold $\theta$, **abstain** and refer the user to a human. Choosing $\theta$ trades **coverage** (answered queries) against **accuracy on answered queries**. **Answers are placeholders; replace them with your department's real procedures.**

In [ ]:
#@title 📚 FAQ knowledge base, and dev/test queries (typos and out-of-scope included)
# The FAQ entries the assistant can answer
FAQ = [
 "course registration opens on the student portal during the first two weeks of the semester",
 "to add or drop a course submit the form to your level adviser before the deadline",
 "school fees are paid through the approved bank channels and the receipt is uploaded on the portal",
 "continuous assessment scores are posted on the department notice board and the portal",
 "the examination timetable is published on the notice board two weeks before the examinations",
 "lecturers hold consultation hours listed on the course outline and you should book by email",
 "the library is open on weekdays and students must carry their identity card to enter",
 "hostel allocation is handled by the student affairs office through the online application",
 "a student who misses an examination because of illness must submit a medical report within one week",
 "the final year project proposal is submitted to the project coordinator and approved by a supervisor",
 "wireless internet access on campus requires registration of your device with the ICT unit",
 "transcript requests are submitted to the registry with the payment receipt and take about two weeks",
]
# Development queries (used to tune the threshold) with the correct FAQ index
DEV = [("how do i register my courses", 0), ("when can i add or drop a course", 1), ("where do i pay school fees", 2), ("where are my CA scores posted", 3),
       ("exam timetable please", 4), ("how do i meet my lecturer", 5), ("library opening days", 6), ("apply for hostel", 7),
       ("i was sick during the exam what do i do", 8), ("submit project proposal", 9), ("connect my phone to campus wifi", 10), ("how long does a transcript take", 11),
       ("who won the football match", None), ("best jollof recipe", None), ("tell me a joke", None), ("bitcoin price today", None)]
# Test queries (touched only once, for the final score), some with typos
TEST = [("course registraton please", 0), ("drop a coures", 1), ("paying fees receipt", 2), ("when are CA results out", 3), ("timetable for examinations", 4),
        ("consultation with lecturer by email", 5), ("can i enter the library without id", 6), ("hostel application online", 7), ("medical report for missed exam", 8),
        ("who supervises the final year project", 9), ("register device for wifi", 10), ("transcript request payment", 11),
        ("what is the weather in lagos", None), ("who is the president of ghana", None), ("how to cook rice", None), ("latest movie", None),
        ("is there a gym on campus", None), ("how do i apply for a visa", None)]
# Show the sizes of the three sets
print(len(FAQ), "FAQ entries;", len(DEV), "dev queries;", len(TEST), "test queries")

### 🔍 Code walkthrough — a pipeline with an exit
The assistant has four stages: **spell-correct** the query, **retrieve** the closest FAQ entry with TF-IDF cosine similarity, **compare** the best score with a threshold θ, then **answer or abstain**. Abstaining is a feature: a wrong confident answer is worse than "I don't know, please contact the office". The threshold is tuned on the DEV set and the TEST set is looked at once, so the final number is honest.

In [ ]:
#@title ⚙️ Spell-correction, retrieval and abstention (TODO: answer and tune_threshold)
def edit1(a, b):
    """True iff Levenshtein distance(a, b) <= 1 (fast check)."""
    # Two words more than one letter apart cannot be one edit apart
    if abs(len(a) - len(b)) > 1: return False
    # Same length: at most one substitution, or one swap of neighbouring letters
    if len(a) == len(b): return sum(x != y for x, y in zip(a, b)) <= 1 or (len(a) > 1 and any(a[:i] + a[i+1] + a[i] + a[i+2:] == b for i in range(len(a) - 1)))
    # Different lengths: s = shorter word, l = longer word
    s, l = (a, b) if len(a) < len(b) else (b, a)
    # One deletion from the longer word gives the shorter word
    return any(l[:i] + l[i+1:] == s for i in range(len(l)))

# Vocabulary of known words from the FAQ
VOCAB = sorted({w for d in FAQ for w in re.findall(r"[a-z]+", d)})
# Spell-correct a query
def correct(text):
    # Output words
    out = []
    # For each word in the query
    for w in re.findall(r"[a-z]+", text.lower()):
        # Keep known and very short words; otherwise replace with a vocabulary word one edit away
        out.append(w if w in VOCAB or len(w) < 4 else next((v for v in VOCAB if edit1(w, v)), w))
    # Join the corrected words
    return " ".join(out)

# Fit TF-IDF on the FAQ and vectorise it
vec = TfidfVectorizer(stop_words="english").fit(FAQ); M = vec.transform(FAQ)
# Similarity of a question to every FAQ entry
def scores(q):
    # Cosine similarity with the FAQ matrix (rows are unit vectors)
    return (vec.transform([correct(q)]) @ M.T).toarray().ravel()

def answer(q, theta):
    """TODO(Student): return the index of the best FAQ entry if its cosine score >= theta, else None (abstain)."""
    # Placeholder (replace with your code)
    return None

def evaluate(data, theta):
    """TODO(Student): return (coverage, accuracy_on_answered, overall_accuracy) where
       coverage = fraction of queries answered (not None); accuracy_on_answered = correct / answered (0 if none answered);
       overall = fraction of ALL queries handled correctly (a None prediction is correct iff gold is None)."""
    # Placeholder for coverage, accuracy on answered, overall accuracy
    return 0.0, 0.0, 0.0

def tune_threshold(dev, grid):
    """TODO(Student): return the theta in grid with the highest OVERALL accuracy on dev (ties → the larger theta)."""
    # Placeholder threshold
    return grid[0]

In [ ]:
#@title ✅ Auto-check, tuning on DEV and a single final look at TEST
check("spelling correction repairs 'registraton' → 'registration'", lambda: "registration" in correct("course registraton"))
check("θ = 0 always answers (coverage 1)",               lambda: evaluate(DEV, 0.0)[0] == 1.0)
check("θ = 2 never answers (coverage 0)",                lambda: evaluate(DEV, 2.0)[0] == 0.0)
check("answer() returns None below threshold",           lambda: answer("tell me a joke", 0.5) is None)
check("answer() finds the fee entry",                    lambda: answer("where do i pay school fees", 0.1) == 2)
# Threshold values to try and the chosen one
grid = np.round(np.arange(0.0, 0.61, 0.05), 2); theta = tune_threshold(DEV, grid)
# Metrics for each threshold
rows = [(t, *evaluate(DEV, t)) for t in grid]
# Plot coverage and accuracy against the threshold
plt.figure(figsize=(5.5, 3.3)); plt.plot(grid, [r[1] for r in rows], label="coverage"); plt.plot(grid, [r[2] for r in rows], label="accuracy on answered"); plt.plot(grid, [r[3] for r in rows], label="overall")
# Mark the chosen threshold
plt.axvline(theta, ls=":", c="k"); plt.xlabel("threshold θ"); plt.legend(); plt.show()
# Final TEST evaluation with the chosen threshold
cov, acc_ans, overall = evaluate(TEST, theta); n = len(TEST); k = int(round(overall * n))
# Wilson 95% CI for the overall accuracy
ci = stats.binomtest(k, n).proportion_ci(0.95, method="wilson")
# Print the TEST results
print(f"θ* = {theta} (chosen on DEV) → TEST: coverage {cov:.2f}, accuracy on answered {acc_ans:.2f}, overall {overall:.2f} (95% Wilson CI [{ci.low:.2f}, {ci.high:.2f}], n = {n})")
check("overall test accuracy ≥ 0.75",                    lambda: overall >= 0.75)
check("tuned θ beats the always-answer baseline on TEST", lambda: overall > evaluate(TEST, 0.0)[2])

**Discuss.** (1) The test set is small ($n=18$): how wide is the CI, and what would you do about it? (2) We tuned $\theta$ on DEV and looked at TEST **once** — why does that matter? (3) Which is worse for a *real* student: a wrong confident answer or an unnecessary referral? How would you move $\theta$?

## Part B — Dialect-bias audit: English vs Nigerian Pidgin
A sentiment model trained or designed on standard English can fail on Pidgin. We audit a lexicon classifier on **16 English and 16 Pidgin sentences** (balanced positive/negative). *Toy data written for teaching — a Pidgin speaker should review and extend it before any real use.* Metrics: accuracy per group, the **gap** $\Delta=\mathrm{acc}_{\text{Eng}}-\mathrm{acc}_{\text{Pid}}$, a **Fisher exact test** and a **bootstrap CI** for $\Delta$.

In [ ]:
#@title 🗂️ Audit data and the English-only lexicon classifier
# English sentences with sentiment labels
EN = [("the lecture was very good", 1), ("i love this course", 1), ("the teacher is helpful and kind", 1), ("this app is great and easy to use", 1),
      ("the results are excellent", 1), ("we enjoyed the practical class", 1), ("the library is nice and quiet", 1), ("i am happy with my grade", 1),
      ("the lecture was boring and slow", 0), ("i hate the new timetable", 0), ("the network is terrible today", 0), ("this fee is a waste of money", 0),
      ("the exam was difficult and unfair", 0), ("the service is poor", 0), ("i am sad about my result", 0), ("the hostel is bad", 0)]
# Nigerian Pidgin sentences
PID = [("the lecture sweet well well", 1), ("na correct teacher be that one", 1), ("this app don make my work easy", 1), ("oga lecturer explain am well", 1),
       ("my result fine and i dey happy", 1), ("the class dey enjoyable", 1), ("e good die", 1), ("dem treat us well", 1),
       ("wahala dey for hostel", 0), ("network bad pass today", 0), ("dem don waste my time", 0), ("this fee too cost e pain me", 0),
       ("the exam don tire me", 0), ("i no like how dem teach am", 0), ("the lecture no make sense", 0), ("light no dey and i dey vex", 0)]
# Positive word list
POS = {"good", "great", "excellent", "love", "happy", "easy", "enjoy", "enjoyed", "enjoyable", "helpful", "fine", "nice", "kind"}
# Negative word list
NEG = {"bad", "terrible", "hate", "waste", "difficult", "slow", "poor", "boring", "sad", "unfair", "pain", "vex"}
# Lexicon classifier: positive count minus negative count
def lex_predict(s, pos=POS, neg=NEG):
    # Words in the sentence and the score
    t = set(re.findall(r"[a-z]+", s.lower())); sc = len(t & pos) - len(t & neg)
    # Ties default to positive
    return 1 if sc >= 0 else 0                                            # ties default to positive
# Print the data sizes
print(len(EN), "English and", len(PID), "Pidgin sentences")

In [ ]:
#@title 📊 group_metrics and the gap (TODO)
def group_metrics(preds, gold, groups):
    """TODO(Student): return a dict {group: accuracy} for each distinct value in `groups`."""
    # Placeholder (replace)
    return {}

def gap_ci(correct_a, correct_b, B=5000, seed=42):
    """TODO(Student): bootstrap 95% CI for mean(correct_a) − mean(correct_b); resample each group with replacement B times.
       Return (gap, lo, hi)."""
    # Placeholder for coverage, accuracy on answered, overall accuracy
    return 0.0, 0.0, 0.0

In [ ]:
#@title ✅ Auto-check and audit
check("group_metrics example", lambda: group_metrics([1, 0, 1, 1], [1, 1, 1, 0], ["a", "a", "b", "b"]) == {"a": 0.5, "b": 0.5})
check("gap_ci on identical groups straddles 0", lambda: gap_ci([1, 0, 1, 0], [1, 0, 1, 0])[1] <= 0 <= gap_ci([1, 0, 1, 0], [1, 0, 1, 0])[2])
# Combine English and Pidgin with a group label
data = [(s, y, "English") for s, y in EN] + [(s, y, "Pidgin") for s, y in PID]
# Predict, collect gold labels and groups
preds = [lex_predict(s) for s, _, _ in data]; gold = [y for _, y, _ in data]; groups = [g for _, _, g in data]
# Print accuracy per group
acc = group_metrics(preds, gold, groups); print({k: round(v, 3) for k, v in acc.items()})
# Correctness arrays for each group
ce = np.array([p == y for p, y, g in zip(preds, gold, groups) if g == "English"]); cp = np.array([p == y for p, y, g in zip(preds, gold, groups) if g == "Pidgin"])
# Gap with bootstrap CI and Fisher exact test
gap, lo, hi = gap_ci(ce, cp); fe = stats.fisher_exact([[ce.sum(), (~ce).sum()], [cp.sum(), (~cp).sum()]])
# Print the gap, CI and p-value
print(f"gap Δ = {gap:.3f}  (bootstrap 95% CI [{lo:.3f}, {hi:.3f}]); Fisher exact p = {fe[1]:.4f}")
# mitigation: extend the lexicon with Pidgin words (a human-in-the-loop fix) and re-audit
# Add Pidgin words to both lexicons
POS2, NEG2 = POS | {"sweet", "correct", "well", "fine"}, NEG | {"wahala", "tire", "vex", "cost", "no"}
# Predict again and compare accuracies
preds2 = [lex_predict(s, POS2, NEG2) for s, _, _ in data]; acc2 = group_metrics(preds2, gold, groups); print("after adding Pidgin words:", {k: round(v, 3) for k, v in acc2.items()})
check("the English-only lexicon is worse on Pidgin than on English", lambda: acc["Pidgin"] < acc["English"])
check("the extended lexicon narrows the gap", lambda: abs(acc2["English"] - acc2["Pidgin"]) < abs(acc["English"] - acc["Pidgin"]))

**Discuss.** Extending the lexicon by hand fixed *these* sentences — but we fitted the fix to the audit data. What would a fair re-test need? Who should write and validate Pidgin (and Yorùbá, Igbo, Hausa) test sets, and how should they be credited and compensated?

## Part C — Responsible-AI checklist for a proposed system
Imagine your department deploys the campus assistant of Part A for 5,000 students. Complete the checklist in your own words (**≥ 20 characters each**). Think of *who is harmed if it is wrong*, *what data it stores*, and *who is accountable*.

In [ ]:
#@title 📝 Your checklist
# Final project checklist
CHECKLIST = {
    "purpose_and_scope":  "",   # what the assistant may and may not answer
    "data_and_privacy":   "",   # what queries are stored, for how long, who can read them (NDPA 2023)
    "fairness_languages": "",   # which languages/dialects are supported; how performance gaps are monitored
    "errors_and_harm":    "",   # worst plausible wrong answer and the safeguard (e.g. abstention, human referral)
    "transparency":       "",   # how users know they are talking to a machine; explanation of answers
    "accountability":     "",   # named owner, complaint channel, review cycle
}

In [ ]:
#@title ✅ Auto-check
check("all six sections are completed (≥ 20 characters)", lambda: all(len(v.strip()) >= 20 for v in CHECKLIST.values()))
check("the checklist names a concrete safeguard for errors (mentions human/refer/abstain/source)", lambda: any(w in CHECKLIST["errors_and_harm"].lower() for w in ("human", "refer", "abstain", "abstention", "source", "review")))

## Part D — Mock practical (60 minutes, closed-book except course notebooks)
**Mirrors the examination structure:** Section A (20 marks, short answers), Section B (30 marks, two of three structured questions), Section C (20 marks, practical). Total 70.

**Section C — practical (do in a fresh notebook, seed 42):**
1. *(6 marks)* Implement A\* on the 8-puzzle with Manhattan distance; report nodes expanded for 5 random instances (mean ± sd) and compare with uniform-cost search.
2. *(6 marks)* Convert *"Every student who attends lectures passes the course"* and *"Ada attends lectures"* to FOL and prove *"Ada passes the course"* by resolution; show the clauses.
3. *(4 marks)* Build a TF-IDF retriever over five sentences you write and evaluate top-1 accuracy on five questions.
4. *(4 marks)* State one fairness risk of your retriever and one mitigation.

**Marking aid:** correctness 50% · evaluation and statistics 20% · code quality and reproducibility 15% · interpretation 15%.

**Section A/B practice questions (answers in the Instructor folder):**
1. Define AI under the four approaches and give one example of each. *(Bloom: Remember/Understand)*
2. Distinguish the Turing test from the Chinese Room argument. *(Understand/Analyse)*
3. State when A\* is optimal; prove that an admissible heuristic is never overestimating, and give an example of an inadmissible one. *(Analyse)*
4. Compare BFS, DFS and iterative deepening on completeness, optimality, time and space. *(Analyse)*
5. Skolemise $\forall x\,\exists y\,\mathrm{Loves}(x,y)$ and explain why the order of quantifiers matters. *(Apply/Analyse)*
6. Describe the components of an expert system and show one forward-chaining step. *(Understand/Apply)*
7. Why does combinatorial explosion make brute-force search impractical for the travelling-salesperson problem on 20 cities? Compute the number of tours. *(Apply)*

## 📝 Assignment — final mini-project (10 marks, individual, 2-page report + notebook)
Adapt the campus assistant to **your own FAQ domain** (a department, a hostel, a clinic or a market):
| Criterion | Marks |
|---|---|
| ≥ 12 FAQ entries, a DEV set (≥ 12 queries) and a **separate** TEST set (≥ 12 queries including typos and out-of-scope questions) | 2 |
| Abstention threshold chosen on DEV only; TEST reported once: coverage, accuracy on answered, overall accuracy with 95% Wilson CI | 3 |
| Dialect audit with **10 or more Pidgin or Yorùbá queries**: group accuracy, gap with bootstrap CI | 2 |
| Completed responsible-AI checklist with a concrete safeguard for each section | 2 |
| Clarity, reproducibility (seed 42) and honest limitations | 1 |

*Marked with the course rubric (class note, Table 0.3): correctness, evidence (numbers with CI and a named test), code clarity, interpretation. Submit the **executed notebook** (Runtime → Run all, outputs visible) as a Colab share link or `.ipynb` before the next lab.*

## Reflection
Write 5 sentences on what you can now do that you could not in Week 1, one thing you still find hard, and one way you will use AI responsibly in your own projects.

```text
Write your reflection here…
```

## Extensions
- Replace TF-IDF by a small sentence-embedding model and compare coverage and accuracy at the same abstention threshold.
- Add a **human hand-off** message and measure how often a real student would need it.
- Write a one-paragraph *data statement* for your FAQ set: who wrote it, in which languages, what is missing.

In [ ]:
checks_summary()  # how many auto-checks have you passed?